In [1]:
# ============================================================
# HYBRID ARCHITECTURE: MultiScale-CNN+SE+Spectral (feature
# extraction) + BiMamba (context modeling) -- PLAIN CE ONLY
# (No SupCon, No artifact weighting yet -- added step by step)
#
# Motivation:
#   Prior ablation showed:
#     MultiScale-CNN + BiLSTM+Transformer (plain) -> F1=0.7955
#     BiT-MamSleep (triple-CNN, no SE/spectral) + BiMamba
#       + SupCon + artifact (ensemble)            -> F1=0.8016
#   Since BiLSTM+Transformer variants (with or without
#   SupCon/artifact) all plateaued around 0.793-0.795, while
#   your original BiMamba-based pipeline reached 0.80+, the
#   context-modeling component (BiMamba vs BiLSTM+Transformer)
#   appears to matter more than the loss design on this dataset.
#
#   This script tests the hybrid: keep the RICHER feature
#   extractor (multi-scale CNN + SE + FFT spectral branch) but
#   swap the context modeling back to BiMamba, with plain CE
#   loss first, so any change vs. 0.8016 can be attributed to
#   the richer front-end alone before re-adding SupCon+artifact.
#
# Ablation plan for this hybrid:
#   1. Hybrid, plain CE (this script)              -> F1 = ?
#   2. Hybrid + artifact-aware only                 -> F1 = ?
#   3. Hybrid + SupCon only                         -> F1 = ?
#   4. Hybrid + SupCon + artifact (both)             -> F1 = ?
#   5. Hybrid + SupCon + artifact, 5-seed ensemble   -> F1 = ?
# ============================================================

import os
import csv
import math
import random
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score, confusion_matrix
)
warnings.filterwarnings('ignore')

# ============================================================
# PATHS & CONFIG
# ============================================================
SAVE_PATH = r"D:\22\AA\preprocess\preprocessed_FFinal"
EVAL_PATH = r"D:\22\AA\evaluation\hybrid_multiscale_bimamba_plain_c7"
os.makedirs(EVAL_PATH, exist_ok=True)

LABEL_NAMES = ["Wake", "N1", "N2", "N3", "REM"]
CONTEXT     = 7
WINDOW      = 2 * CONTEXT + 1
N_EPOCHS    = 30
BATCH_SIZE  = 64
SEEDS       = [42, 123, 256, 789, 999]

D_MODEL = 128
D_STATE = 16     # Mamba state dimension
DROPOUT = 0.4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device       : {device}")
print(f"Architecture : HYBRID = MultiScale-CNN+SE+Spectral -> BiMamba")
print(f"Loss         : PLAIN weighted CE (NO SupCon, NO artifact weighting)")
print(f"Context      : {CONTEXT} (window={WINDOW})")
print(f"Output       : {EVAL_PATH}")


# ============================================================
# FIXED SPLIT
# ============================================================
_train_path = os.path.join(SAVE_PATH, "_train_subs.npy")
_test_path  = os.path.join(SAVE_PATH, "_test_subs.npy")
TRAIN_SUBS = np.load(_train_path, allow_pickle=True).tolist()
TEST_SUBS  = np.load(_test_path,  allow_pickle=True).tolist()
print(f"Split loaded -> Train:{len(TRAIN_SUBS)}  Test:{len(TEST_SUBS)}")


# ============================================================
# DATASET (plain -- no artifact weight)
# ============================================================
class PlainDataset(Dataset):
    def __init__(self, subject_list, data_path, context=CONTEXT):
        self.context = context
        self.data  = []
        self.index = []

        label_counter = Counter()
        for sub in subject_list:
            fp = os.path.join(data_path, f"{sub}.npz")
            if not os.path.exists(fp):
                continue
            with np.load(fp) as d:
                eeg    = d['eeg'][:, [0, 1], :]
                eog    = d['eog'][:, [0], :]
                signal = np.concatenate([eeg, eog], axis=1).astype(np.float32)
                labels = d['labels'].copy()

            n = len(labels)
            sub_idx = len(self.data)
            self.data.append((signal, labels))
            for i in range(n):
                self.index.append((sub_idx, i, n))
                label_counter[int(labels[i])] += 1

        self.label_counts = np.array(
            [label_counter[i] for i in range(5)], dtype=np.float32
        )
        total = len(self.index)
        ram = sum(s.nbytes for s, _ in self.data) / 1e9
        print(f"  Samples: {total:,}   RAM: {ram:.2f} GB")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        sub_idx, center_i, n = self.index[idx]
        signal, labels = self.data[sub_idx]
        window_epochs = []
        for offset in range(-self.context, self.context + 1):
            ei = max(0, min(n - 1, center_i + offset))
            window_epochs.append(signal[ei])
        x = np.stack(window_epochs, axis=0)
        y = int(labels[center_i])
        return torch.FloatTensor(x), torch.tensor(y, dtype=torch.long)


print("\nBuilding datasets...")
train_ds = PlainDataset(TRAIN_SUBS, SAVE_PATH)
print()
test_ds  = PlainDataset(TEST_SUBS, SAVE_PATH)
print("Datasets ready.")


# ============================================================
# FEATURE EXTRACTOR: MultiScale CNN + FFT SPECTRAL BRANCH + SE
# (identical to the MultiScaleSleepNet-inspired script)
# ============================================================
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.shape
        s = self.pool(x).view(b, c)
        s = self.fc(s).view(b, c, 1)
        return x * s


class MultiScaleCNN(nn.Module):
    """Small/medium/large kernel branches + FFT spectral branch + SE."""
    def __init__(self, in_ch=3, d_model=D_MODEL, dropout=DROPOUT):
        super().__init__()
        mid = d_model // 4

        def time_branch(kernel):
            return nn.Sequential(
                nn.Conv1d(in_ch, mid, kernel_size=kernel, stride=6, padding=kernel // 2),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
                nn.Conv1d(mid, mid, kernel_size=8, padding=4),
                nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
                nn.Dropout(dropout),
            )

        self.small  = time_branch(25)
        self.medium = time_branch(50)
        self.large  = time_branch(100)

        self.spectral = nn.Sequential(
            nn.Conv1d(in_ch, mid, kernel_size=25, stride=6, padding=12),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(4, 4),
            nn.Conv1d(mid, mid, kernel_size=8, padding=4),
            nn.BatchNorm1d(mid), nn.GELU(), nn.MaxPool1d(2, 2),
            nn.Dropout(dropout),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, in_ch, 3000)
            L_s = self.small(dummy).shape[2]
            L_m = self.medium(dummy).shape[2]
            L_l = self.large(dummy).shape[2]
            L_f = self.spectral(dummy).shape[2]

        target_L = min(L_s, L_m, L_l, L_f)
        self.pool_s = nn.AdaptiveAvgPool1d(target_L)
        self.pool_m = nn.AdaptiveAvgPool1d(target_L)
        self.pool_l = nn.AdaptiveAvgPool1d(target_L)
        self.pool_f = nn.AdaptiveAvgPool1d(target_L)
        self.out_len = target_L

        self.se = SEBlock(4 * mid)
        self.proj = nn.Sequential(
            nn.Conv1d(4 * mid, d_model, kernel_size=1),
            nn.BatchNorm1d(d_model), nn.GELU(),
        )

    def forward(self, x):
        fs = self.pool_s(self.small(x))
        fm = self.pool_m(self.medium(x))
        fl = self.pool_l(self.large(x))

        x_fft = torch.fft.rfft(x, dim=-1)
        x_mag = torch.abs(x_fft)
        if x_mag.shape[-1] < x.shape[-1]:
            pad = x.shape[-1] - x_mag.shape[-1]
            x_mag = F.pad(x_mag, (0, pad))
        else:
            x_mag = x_mag[..., :x.shape[-1]]
        ff = self.pool_f(self.spectral(x_mag))

        feat = torch.cat([fs, fm, fl, ff], dim=1)
        feat = self.se(feat)
        return self.proj(feat)


# ============================================================
# CONTEXT MODELING: PURE PYTORCH BIDIRECTIONAL MAMBA (SSM)
# (identical to your BiT-MamSleep script)
# ============================================================
class MambaBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, d_state=D_STATE, d_conv=4, expand=2, dropout=DROPOUT):
        super().__init__()
        self.d_model  = d_model
        self.d_state  = d_state
        self.d_inner  = d_model * expand

        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.conv1d = nn.Conv1d(
            self.d_inner, self.d_inner, kernel_size=d_conv,
            padding=d_conv - 1, groups=self.d_inner, bias=True
        )
        self.x_proj  = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)
        self.dt_proj = nn.Linear(1, self.d_inner, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0)
        A = A.expand(self.d_inner, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))

        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm     = nn.LayerNorm(d_model)
        self.dropout  = nn.Dropout(dropout)

    def ssm_scan(self, x, A, B, C, D):
        B_size, L, d = x.shape
        N = self.d_state
        h = torch.zeros(B_size, d, N, device=x.device, dtype=x.dtype)
        ys = []
        for i in range(L):
            xi = x[:, i, :]
            Bi = B[:, i, :]
            Ci = C[:, i, :]
            h = h * A.unsqueeze(0) + (xi.unsqueeze(-1) * Bi.unsqueeze(1))
            y = (h * Ci.unsqueeze(1)).sum(-1) + D * xi
            ys.append(y)
        return torch.stack(ys, dim=1)

    def forward(self, x):
        residual = x
        B, L, _ = x.shape

        xz    = self.in_proj(x)
        x_, z = xz.chunk(2, dim=-1)

        x_ = x_.transpose(1, 2)
        x_ = self.conv1d(x_)[:, :, :L]
        x_ = x_.transpose(1, 2)
        x_ = F.silu(x_)

        ssm_params = self.x_proj(x_)
        dt_raw, B_param, C_param = ssm_params.split(
            [1, self.d_state, self.d_state], dim=-1
        )
        dt = F.softplus(self.dt_proj(dt_raw))

        A    = -torch.exp(self.A_log)
        A_dt = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))
        A_disc = A_dt.mean(dim=1).mean(dim=0)

        y = self.ssm_scan(x_, A_disc, B_param, C_param, self.D)
        y = y * F.silu(z)
        y = self.out_proj(y)
        y = self.dropout(y)

        return self.norm(y + residual)


class BiMambaBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, d_state=D_STATE, dropout=DROPOUT):
        super().__init__()
        self.forward_mamba  = MambaBlock(d_model, d_state, dropout=dropout)
        self.backward_mamba = MambaBlock(d_model, d_state, dropout=dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        fwd = self.forward_mamba(x)
        x_flip = torch.flip(x, dims=[1])
        bwd     = self.backward_mamba(x_flip)
        bwd     = torch.flip(bwd, dims=[1])
        return self.norm(fwd + bwd)


# ============================================================
# FULL HYBRID MODEL (plain -- no SupCon head yet)
# ============================================================
class HybridMultiScaleBiMamba(nn.Module):
    """
    Feature extractor : MultiScale CNN + FFT spectral + SE
    Context modeling   : Bidirectional Mamba (intra-epoch + inter-epoch)
    Loss (this script) : plain weighted CE
    """
    def __init__(
        self, in_ch=3, d_model=D_MODEL, d_state=D_STATE,
        n_layers=2, dropout=DROPOUT, n_classes=5, context=CONTEXT
    ):
        super().__init__()
        self.context = context
        self.d_model = d_model

        self.cnn = MultiScaleCNN(in_ch=in_ch, d_model=d_model, dropout=dropout)
        self.intra_seq_len = self.cnn.out_len

        self.intra_bimamba = nn.Sequential(*[
            BiMambaBlock(d_model, d_state, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.inter_pos = nn.Parameter(torch.randn(1, WINDOW, d_model) * 0.01)
        self.inter_bimamba = nn.Sequential(*[
            BiMambaBlock(d_model, d_state, dropout=dropout)
            for _ in range(n_layers)
        ])

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes),
        )
        # NOTE: no SupCon projection head yet -- add in the next
        # ablation step, same as MultiScaleSleepNetSupCon did.

    def forward(self, x):
        # x: (B, W, C, T)
        B, W, C, T = x.shape

        cnn_out = self.cnn(x.view(B * W, C, T))          # (B*W, d_model, L)
        cnn_out = cnn_out.permute(0, 2, 1)                # (B*W, L, d_model)
        intra   = self.intra_bimamba(cnn_out)

        epoch_feat = intra.mean(dim=1).view(B, W, self.d_model)

        inter = epoch_feat + self.inter_pos
        inter = self.inter_bimamba(inter)
        center = inter[:, self.context, :]

        logits = self.classifier(center)
        return logits


# ============================================================
# CLASS WEIGHTS
# ============================================================
cw_np = train_ds.label_counts.sum() / (5 * train_ds.label_counts)
cw = torch.FloatTensor(cw_np).to(device)
print(f"\nClass weights:")
for name, w in zip(LABEL_NAMES, cw_np):
    print(f"  {name}: {w:.3f}")


def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_epoch_fn(model, loader, optimizer, scheduler, criterion):
    model.train()
    total_loss = 0
    preds, labs = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        preds.extend(logits.argmax(1).cpu().numpy())
        labs.extend(y.cpu().numpy())
    n = len(loader)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    return total_loss / n, acc, f1


def evaluate_fn(model, loader):
    model.eval()
    preds, labs = [], []
    with torch.no_grad():
        for x, y in loader:
            logits = model(x.to(device))
            preds.extend(logits.argmax(1).cpu().numpy())
            labs.extend(y.numpy())
    preds = np.array(preds); labs = np.array(labs)
    acc = accuracy_score(labs, preds)
    f1 = f1_score(labs, preds, average='macro', zero_division=0)
    kappa = cohen_kappa_score(labs, preds)
    per_cls = f1_score(labs, preds, average=None, zero_division=0)
    return acc, f1, kappa, per_cls


# ============================================================
# CSV SETUP
# ============================================================
csv_summary_path = os.path.join(EVAL_PATH, "summary.csv")
fields = ["seed", "acc", "f1_macro", "kappa",
          "f1_Wake", "f1_N1", "f1_N2", "f1_N3", "f1_REM"]
with open(csv_summary_path, 'w', newline='') as f:
    csv.DictWriter(f, fields).writeheader()

# ============================================================
# 5-SEED TRAINING LOOP
# ============================================================
all_results = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}\nSEED {seed}  ({seed_idx+1}/{len(SEEDS)})\n{'='*60}")
    set_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
        generator=torch.Generator().manual_seed(seed)
    )
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    model = HybridMultiScaleBiMamba(in_ch=3).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters : {n_params:,}")

    criterion = nn.CrossEntropyLoss(weight=cw)

    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=3e-4,
                             betas=(0.9, 0.98), eps=1e-9)
    total_steps = N_EPOCHS * len(train_loader)
    warmup_steps = int(0.05 * total_steps)

    def warmup_cosine(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        t = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * t))

    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=warmup_cosine)

    best_f1 = 0.0
    best_path = os.path.join(EVAL_PATH, f"best_seed{seed}.pt")

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc, tr_f1 = train_epoch_fn(
            model, train_loader, optimizer, scheduler, criterion
        )
        vl_acc, vl_f1, vl_kap, vl_per = evaluate_fn(model, test_loader)

        saved = ""
        if vl_f1 > best_f1:
            best_f1 = vl_f1
            torch.save(model.state_dict(), best_path)
            saved = " <- BEST"

        lr = optimizer.param_groups[0]['lr']
        print(f"  Ep[{epoch:02d}/{N_EPOCHS}] Loss:{tr_loss:.3f} "
              f"TrAcc:{tr_acc:.3f} ValAcc:{vl_acc:.3f} F1:{vl_f1:.3f} "
              f"k:{vl_kap:.3f} LR:{lr:.2e}{saved}")

    model.load_state_dict(torch.load(best_path, map_location=device))
    fin_acc, fin_f1, fin_kap, fin_per = evaluate_fn(model, test_loader)

    print(f"\n  Seed {seed} FINAL: Acc={fin_acc*100:.2f}% F1={fin_f1:.4f} k={fin_kap:.4f}")
    for i, name in enumerate(LABEL_NAMES):
        print(f"    {name:6s}: {fin_per[i]:.4f}")

    all_results.append({
        'seed': seed, 'acc': fin_acc, 'f1': fin_f1, 'kappa': fin_kap,
        'per_cls': fin_per
    })

    with open(csv_summary_path, 'a', newline='') as f:
        csv.DictWriter(f, fields).writerow({
            "seed": seed,
            "acc": round(fin_acc, 4), "f1_macro": round(fin_f1, 4),
            "kappa": round(fin_kap, 4),
            "f1_Wake": round(fin_per[0], 4), "f1_N1": round(fin_per[1], 4),
            "f1_N2": round(fin_per[2], 4), "f1_N3": round(fin_per[3], 4),
            "f1_REM": round(fin_per[4], 4),
        })

# ============================================================
# FINAL REPORT
# ============================================================
accs   = np.array([r['acc'] for r in all_results]) * 100
f1s    = np.array([r['f1'] for r in all_results])
kappas = np.array([r['kappa'] for r in all_results])
n1s    = np.array([r['per_cls'][1] for r in all_results])

print(f"\n{'='*60}\nHYBRID (MultiScale-CNN+SE+Spectral -> BiMamba), PLAIN CE -- 5-SEED REPORT\n{'='*60}")
print(f"Accuracy : {accs.mean():.2f} +- {accs.std():.2f}%")
print(f"Macro F1 : {f1s.mean():.4f} +- {f1s.std():.4f}")
print(f"Kappa    : {kappas.mean():.4f} +- {kappas.std():.4f}")
print(f"N1 F1    : {n1s.mean():.4f} +- {n1s.std():.4f}")

print(f"\n{'='*60}\nCOMPARISON TABLE\n{'='*60}")
print(f"{'Model':<58} {'Acc%':>7} {'F1':>7}")
print("-" * 80)
print(f"{'MultiScale+BiLSTM/Transformer, plain':<58} {'83.53':>7} {'0.7955':>7}")
print(f"{'MultiScale+BiLSTM/Transformer, +artifact':<58} {'83.31':>7} {'0.7933':>7}")
print(f"{'MultiScale+BiLSTM/Transformer, +SupCon':<58} {'83.24':>7} {'0.7934':>7}")
print(f"{'Original BiT-MamSleep (triple-CNN), plain':<58} {'~82-83':>7} {'~0.79':>7}")
print(f"{'Original BiT-MamSleep + SupCon+artifact (ensemble)':<58} {'84.01':>7} {'0.8016':>7}")
print(f"{'HYBRID: MultiScale-CNN + BiMamba, plain (this)':<58} {accs.mean():>7.2f} {f1s.mean():>7.4f}")
print(f"{'='*80}")



print(f"\nSummary saved: {csv_summary_path}")
print("Done!")

Device       : cuda
Architecture : HYBRID = MultiScale-CNN+SE+Spectral -> BiMamba
Loss         : PLAIN weighted CE (NO SupCon, NO artifact weighting)
Context      : 7 (window=15)
Output       : D:\22\AA\evaluation\hybrid_multiscale_bimamba_plain_c7
Split loaded -> Train:76  Test:20

Building datasets...
  Samples: 71,347   RAM: 2.57 GB

  Samples: 19,763   RAM: 0.71 GB
Datasets ready.

Class weights:
  Wake: 1.844
  N1: 3.142
  N2: 0.442
  N3: 1.016
  REM: 1.117

SEED 42  (1/5)
  Parameters : 990,741
  Ep[01/30] Loss:0.860 TrAcc:0.683 ValAcc:0.738 F1:0.698 k:0.653 LR:3.33e-04 <- BEST
  Ep[02/30] Loss:0.512 TrAcc:0.815 ValAcc:0.801 F1:0.761 k:0.721 LR:5.00e-04 <- BEST
  Ep[03/30] Loss:0.448 TrAcc:0.829 ValAcc:0.809 F1:0.761 k:0.733 LR:4.97e-04 <- BEST
  Ep[04/30] Loss:0.412 TrAcc:0.840 ValAcc:0.830 F1:0.789 k:0.762 LR:4.91e-04 <- BEST
  Ep[05/30] Loss:0.383 TrAcc:0.846 ValAcc:0.810 F1:0.771 k:0.737 LR:4.82e-04
  Ep[06/30] Loss:0.351 TrAcc:0.856 ValAcc:0.809 F1:0.762 k:0.737 LR:4.70e-04


KeyboardInterrupt: 